# Sentence Transformers (SBERT)

## Goal
Demonstrate how **sentence-transformers** (SBERT) produces dense semantic embeddings for sentences and documents — and how to use them for practical NLP tasks.

**Key questions we'll answer:**
- What makes SBERT different from word vectors or vanilla BERT?
- How do we compute semantic similarity between sentences?
- How do we build a simple semantic search / retrieval system?
- How do SBERT embeddings cluster by topic?

**Models used**: `all-MiniLM-L6-v2` (fast, small) and `all-mpnet-base-v2` (more accurate)  
**Library**: [`sentence-transformers`](https://www.sbert.net/) from UKP Lab

## Setup

In [ ]:
# Uncomment to install missing packages
# !pip install sentence-transformers scikit-learn matplotlib seaborn pandas

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sentence_transformers import SentenceTransformer, util
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

print('All imports OK')

---
## 1. What is SBERT?

**BERT** produces contextualized token embeddings, but to get a single sentence vector you need to pool them (e.g. [CLS] token or mean-pool). Naive BERT is **very slow** for similarity search: comparing N sentences requires N² forward passes.

**SBERT** fine-tunes BERT with a **siamese / triplet network** objective so that the pooled sentence vector *directly* captures semantic meaning. Comparing two sentences is now just a **cosine similarity** between two cached vectors — orders of magnitude faster.

```
sentence → BERT encoder → mean-pool → 384-dim or 768-dim vector
```

Use-cases: semantic search, clustering, paraphrase detection, RAG retrieval.

---
## 2. Loading a Model

In [ ]:
# all-MiniLM-L6-v2: 22M params, 384-dim embeddings, very fast
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f'Model max sequence length: {model.max_seq_length} tokens')
print(f'Embedding dimension: {model.get_sentence_embedding_dimension()}')

---
## 3. Encoding Sentences

In [ ]:
sentences = [
    'The cat sat on the mat.',
    'A kitten rested on the rug.',
    'The dog barked at the mailman.',
    'Machine learning is a subset of artificial intelligence.',
    'Deep learning uses neural networks with many layers.',
]

# encode() returns a numpy array of shape (N, dim)
embeddings = model.encode(sentences, show_progress_bar=False)
print(f'Embeddings shape: {embeddings.shape}')  # (5, 384)

---
## 4. Semantic Similarity

The `util.cos_sim` helper computes a full cosine similarity matrix.

In [ ]:
sim_matrix = util.cos_sim(embeddings, embeddings).numpy()

short_labels = [s[:30] + '...' if len(s) > 30 else s for s in sentences]

plt.figure(figsize=(8, 6))
sns.heatmap(
    sim_matrix,
    annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=short_labels, yticklabels=short_labels
)
plt.title('Cosine Similarity Matrix')
plt.tight_layout()
plt.show()

**Expected pattern**: sentences 0 & 1 (both about cats/kittens) should have high similarity; sentences 3 & 4 (both about ML/DL) should also cluster together.

---
## 5. Sentence Pair Comparison

Given two lists A and B, find the most similar pairs.

In [ ]:
queries = [
    'What is machine learning?',
    'Tell me about domestic animals.',
    'How do neural networks learn?',
]

corpus = [
    'Machine learning algorithms improve through experience.',
    'Cats and dogs are popular household pets.',
    'Neural networks adjust their weights via backpropagation.',
    'The stock market fluctuated yesterday.',
    'Supervised learning requires labeled training data.',
]

query_emb  = model.encode(queries,  convert_to_tensor=True)
corpus_emb = model.encode(corpus,   convert_to_tensor=True)

# For each query, retrieve top-2 corpus sentences
hits = util.semantic_search(query_emb, corpus_emb, top_k=2)

for q_idx, query in enumerate(queries):
    print(f'\nQuery: "{query}"')
    for hit in hits[q_idx]:
        print(f'  [{hit["score"]:.3f}] {corpus[hit["corpus_id"]]}')

---
## 6. Semantic Search over a Larger Corpus

A realistic demo: encode a corpus once, then answer multiple queries efficiently.

In [ ]:
wiki_snippets = [
    # Space
    'The Moon is Earth\'s only natural satellite and the fifth-largest in the Solar System.',
    'Mars is the fourth planet from the Sun and is often called the Red Planet.',
    'The James Webb Space Telescope observes the universe in infrared light.',
    'Black holes have gravitational pulls so strong that nothing, not even light, can escape.',
    # Biology
    'DNA carries genetic information in all living organisms and many viruses.',
    'Photosynthesis is the process by which plants convert sunlight into chemical energy.',
    'The human brain contains approximately 86 billion neurons.',
    'CRISPR-Cas9 is a gene-editing technology that can modify DNA sequences.',
    # History
    'The Roman Empire reached its greatest extent under Emperor Trajan in 117 AD.',
    'The French Revolution began in 1789 and led to the rise of Napoleon Bonaparte.',
    'The Great Wall of China was built over many centuries to protect against invasions.',
    'World War II ended in 1945 after Germany and Japan surrendered.',
    # Technology
    'The transformer architecture, introduced in 2017, underpins modern large language models.',
    'The Internet was publicly launched in 1991 with the World Wide Web.',
    'GPUs were originally designed for graphics but are now essential for deep learning.',
    'The first iPhone was unveiled by Steve Jobs in 2007.',
]

# Encode corpus once — in production you would cache/save this
corpus_embeddings = model.encode(wiki_snippets, convert_to_tensor=True, show_progress_bar=False)
print(f'Corpus encoded: {corpus_embeddings.shape}')

In [ ]:
def semantic_search(query: str, top_k: int = 3):
    """Retrieve the top_k most relevant snippets for a query."""
    q_emb = model.encode(query, convert_to_tensor=True)
    results = util.semantic_search(q_emb, corpus_embeddings, top_k=top_k)[0]
    print(f'Query: "{query}"')
    for r in results:
        print(f'  [{r["score"]:.3f}] {wiki_snippets[r["corpus_id"]]}')
    print()

semantic_search('How do plants make food from sunlight?')
semantic_search('What happened during World War 2?')
semantic_search('Tell me about AI hardware acceleration.')

---
## 7. Paraphrase Detection

Pairs with cosine similarity above a threshold are likely paraphrases.

In [ ]:
pairs = [
    ('How old are you?',          'What is your age?'),
    ('It is raining outside.',    'The weather is wet today.'),
    ('I love pizza.',             'The capital of France is Paris.'),
    ('She runs every morning.',   'She jogs daily before breakfast.'),
    ('Open the door please.',     'Could you unlock the entrance?'),
]

THRESHOLD = 0.75

for a, b in pairs:
    ea, eb = model.encode([a, b])
    score = float(util.cos_sim(ea, eb))
    label = 'PARAPHRASE' if score >= THRESHOLD else 'DIFFERENT'
    print(f'[{score:.3f}] {label}')
    print(f'  A: {a}')
    print(f'  B: {b}')

---
## 8. Clustering Sentences by Topic

K-Means on SBERT embeddings groups semantically related sentences even without labels.

In [ ]:
topic_sentences = [
    # Sports
    'The team scored three goals in the second half.',
    'The athlete broke the world record in the 100-meter sprint.',
    'The basketball player signed a new contract worth millions.',
    'The tournament will be held in Paris next summer.',
    # Cooking
    'Add two cups of flour and mix until smooth.',
    'Preheat the oven to 180 degrees Celsius before baking.',
    'Season the chicken with salt, pepper, and garlic.',
    'Let the dough rest for an hour before shaping.',
    # Finance
    'The stock market rallied after the Federal Reserve announcement.',
    'Inflation has risen to its highest level in forty years.',
    'Investors diversify their portfolios to reduce risk.',
    'The central bank raised interest rates by 25 basis points.',
]

true_labels = ['Sports'] * 4 + ['Cooking'] * 4 + ['Finance'] * 4

embs = model.encode(topic_sentences, show_progress_bar=False)

kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
cluster_ids = kmeans.fit_predict(embs)

df = pd.DataFrame({'sentence': topic_sentences, 'true_topic': true_labels, 'cluster': cluster_ids})
print(df.to_string(index=False))

In [ ]:
# Visualise with t-SNE
tsne = TSNE(n_components=2, perplexity=5, random_state=42)
coords = tsne.fit_transform(embs)

colors = {'Sports': 'steelblue', 'Cooking': 'tomato', 'Finance': 'seagreen'}
markers = {0: 'o', 1: 's', 2: '^'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: ground-truth topics
for topic, color in colors.items():
    mask = [t == topic for t in true_labels]
    axes[0].scatter(coords[mask, 0], coords[mask, 1], c=color, label=topic, s=100, edgecolors='k')
axes[0].set_title('Ground-truth Topics')
axes[0].legend()

# Right: K-Means clusters
for k in range(3):
    mask = cluster_ids == k
    axes[1].scatter(coords[mask, 0], coords[mask, 1], marker=markers[k], label=f'Cluster {k}', s=100, edgecolors='k')
axes[1].set_title('K-Means Clusters (k=3)')
axes[1].legend()

for ax in axes:
    ax.set_xlabel('t-SNE dim 1')
    ax.set_ylabel('t-SNE dim 2')

plt.suptitle('SBERT Embeddings — t-SNE Visualisation', fontsize=13)
plt.tight_layout()
plt.show()

---
## 9. Cross-Encoder vs. Bi-Encoder

SBERT uses a **bi-encoder**: both sentences are encoded independently → fast but approximate.  
A **cross-encoder** encodes the pair jointly → slower but more accurate. Typical production pipeline: bi-encoder for retrieval (top-100), cross-encoder for re-ranking (top-10).

In [ ]:
from sentence_transformers import CrossEncoder

# Lightweight cross-encoder fine-tuned on MS-MARCO
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

query = 'What causes climate change?'
candidates = [
    'Greenhouse gas emissions from burning fossil fuels trap heat in the atmosphere.',
    'Deforestation reduces the number of trees that absorb CO2.',
    'The price of oil dropped significantly last quarter.',
    'Scientists study ocean temperatures to understand global warming.',
    'Renewable energy sources like solar and wind have zero emissions.',
]

# Bi-encoder scores
q_emb = model.encode(query)
c_embs = model.encode(candidates)
bi_scores = cosine_similarity([q_emb], c_embs)[0]

# Cross-encoder scores
cross_scores = cross_encoder.predict([(query, c) for c in candidates])

df_scores = pd.DataFrame({
    'Candidate': candidates,
    'Bi-encoder (cosine)': bi_scores.round(3),
    'Cross-encoder (logit)': cross_scores.round(3),
})

print('=== Bi-encoder ranking ===')
print(df_scores.sort_values('Bi-encoder (cosine)', ascending=False)[['Candidate', 'Bi-encoder (cosine)']].to_string(index=False))

print('\n=== Cross-encoder re-ranking ===')
print(df_scores.sort_values('Cross-encoder (logit)', ascending=False)[['Candidate', 'Cross-encoder (logit)']].to_string(index=False))

---
## 10. Model Comparison: MiniLM vs. MPNet

Bigger models generally produce better embeddings at the cost of speed.

In [ ]:
import time

model_names = [
    'all-MiniLM-L6-v2',    # 22M params, 384-dim — fastest
    'all-mpnet-base-v2',    # 110M params, 768-dim — best quality
]

test_sentences = [
    'Paris is the capital of France.',
    'France\'s capital city is Paris.',
    'The Eiffel Tower is located in Paris.',
    'Berlin is the capital of Germany.',
]

for name in model_names:
    m = SentenceTransformer(name)
    t0 = time.time()
    embs = m.encode(test_sentences)
    elapsed = time.time() - t0
    sim = util.cos_sim(embs[0], embs[1]).item()   # sentence 0 vs 1 (paraphrases)
    diff = util.cos_sim(embs[0], embs[3]).item()  # sentence 0 vs 3 (different city)
    dim = m.get_sentence_embedding_dimension()
    print(f'{name}')
    print(f'  dim={dim}, time={elapsed:.3f}s')
    print(f'  paraphrase sim (0↔1): {sim:.3f}  |  different sim (0↔3): {diff:.3f}')
    print()

---
## 12. Key Takeaways

| Feature | SBERT / Bi-encoder | Cross-encoder |
|---|---|---|
| Speed | Fast (O(1) per query after corpus pre-encoding) | Slow (O(N) per query) |
| Accuracy | Good | Better |
| Use case | Large-scale retrieval | Re-ranking a small candidate set |

**Best practice (RAG pipeline)**:
1. **Index**: encode all documents with SBERT → store vectors
2. **Retrieve**: cosine search → top-K candidates (fast, approximate)
3. **Re-rank**: cross-encoder on top-K → final ranked list (slow, accurate)
4. **Generate**: feed re-ranked context to an LLM

**Useful SBERT models** (from [sbert.net](https://www.sbert.net/docs/sentence_transformer/pretrained_models.html)):

| Model | Dim | Speed | Quality |
|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | Very fast | Good |
| `all-MiniLM-L12-v2` | 384 | Fast | Better |
| `all-mpnet-base-v2` | 768 | Moderate | Best general |
| `multi-qa-mpnet-base-dot-v1` | 768 | Moderate | Best for QA |
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | Fast | 50+ languages |

In [ ]:
# Uncomment to install
# !pip install datasets

from datasets import load_dataset

# Stream a small slice — the full dataset is ~400 k pairs
raw = load_dataset('quora', split='train', trust_remote_code=True)
print(f'Full dataset size: {len(raw):,} pairs')
print(f'Features: {raw.features}')
print(f'\nExample row:\n{raw[0]}')

In [ ]:
# Build a balanced sample: 1000 duplicate pairs + 1000 non-duplicate pairs
import random
random.seed(42)

duplicates     = [r for r in raw if r['is_duplicate'] == 1]
non_duplicates = [r for r in raw if r['is_duplicate'] == 0]

sample = random.sample(duplicates, 1000) + random.sample(non_duplicates, 1000)
random.shuffle(sample)

q1s    = [r['questions']['text'][0] for r in sample]
q2s    = [r['questions']['text'][1] for r in sample]
labels = [r['is_duplicate'] for r in sample]

print(f'Sample size : {len(sample)} pairs')
print(f'Duplicates  : {sum(labels)}')
print(f'Non-dupes   : {len(labels) - sum(labels)}')

# Show a few examples
print('\n--- Duplicate pairs ---')
for r in [s for s in sample if s['is_duplicate'] == 1][:3]:
    print(f'  Q1: {r["questions"]["text"][0]}')
    print(f'  Q2: {r["questions"]["text"][1]}')
    print()

print('--- Non-duplicate pairs ---')
for r in [s for s in sample if s['is_duplicate'] == 0][:3]:
    print(f'  Q1: {r["questions"]["text"][0]}')
    print(f'  Q2: {r["questions"]["text"][1]}')
    print()

In [ ]:
# Encode all questions — encode both lists in one batch for efficiency
print('Encoding questions...')
all_questions = q1s + q2s
all_embeddings = model.encode(all_questions, batch_size=64, show_progress_bar=True)

emb_q1 = all_embeddings[:len(q1s)]
emb_q2 = all_embeddings[len(q1s):]

# Pairwise cosine similarity (diagonal of the full matrix)
similarities = [
    float(util.cos_sim(emb_q1[i], emb_q2[i]))
    for i in range(len(q1s))
]

print(f'\nSimilarity stats:')
print(f'  Duplicates     — mean: {np.mean([s for s, l in zip(similarities, labels) if l==1]):.3f}')
print(f'  Non-duplicates — mean: {np.mean([s for s, l in zip(similarities, labels) if l==0]):.3f}')

In [ ]:
# Similarity distribution: duplicates vs. non-duplicates
sim_dup     = [s for s, l in zip(similarities, labels) if l == 1]
sim_non_dup = [s for s, l in zip(similarities, labels) if l == 0]

plt.figure(figsize=(9, 4))
plt.hist(sim_non_dup, bins=40, alpha=0.6, label='Non-duplicate', color='steelblue')
plt.hist(sim_dup,     bins=40, alpha=0.6, label='Duplicate',     color='tomato')
plt.axvline(x=0.75, color='black', linestyle='--', linewidth=1.2, label='Threshold = 0.75')
plt.xlabel('Cosine Similarity')
plt.ylabel('Count')
plt.title('SBERT Similarity Distribution — Quora Question Pairs')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import (
    precision_recall_curve, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# --- Find the best threshold by maximising F1 ---
sims_arr = np.array(similarities)
labels_arr = np.array(labels)

precisions, recalls, thresholds = precision_recall_curve(labels_arr, sims_arr)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f'Best threshold: {best_threshold:.3f}  (F1 = {f1_scores[best_idx]:.3f})')

# --- Evaluate at the best threshold ---
preds = (sims_arr >= best_threshold).astype(int)
print('\nClassification report:')
print(classification_report(labels_arr, preds, target_names=['Non-duplicate', 'Duplicate']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: Precision-Recall curve with best threshold marked
axes[0].plot(recalls[:-1], precisions[:-1], color='royalblue', linewidth=2)
axes[0].scatter(recalls[best_idx], precisions[best_idx],
                color='red', zorder=5, s=80, label=f'Best threshold ({best_threshold:.2f})')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: Confusion matrix
cm = confusion_matrix(labels_arr, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-dup', 'Duplicate'])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title(f'Confusion Matrix (threshold = {best_threshold:.2f})')

plt.tight_layout()
plt.show()

In [ ]:
# Inspect hard cases: false positives and false negatives
df_results = pd.DataFrame({
    'q1': q1s, 'q2': q2s,
    'label': labels_arr,
    'sim': sims_arr.round(3),
    'pred': preds,
})
df_results['correct'] = df_results['label'] == df_results['pred']

print('=== False Positives (predicted duplicate, actually different) ===')
fp = df_results[(df_results['pred'] == 1) & (df_results['label'] == 0)].head(3)
for _, row in fp.iterrows():
    print(f'  [{row["sim"]:.3f}] Q1: {row["q1"]}')
    print(f'          Q2: {row["q2"]}')
    print()

print('=== False Negatives (predicted different, actually duplicate) ===')
fn = df_results[(df_results['pred'] == 0) & (df_results['label'] == 1)].head(3)
for _, row in fn.iterrows():
    print(f'  [{row["sim"]:.3f}] Q1: {row["q1"]}')
    print(f'          Q2: {row["q2"]}')
    print()

---
## 11. Key Takeaways

| Feature | SBERT / Bi-encoder | Cross-encoder |
|---|---|---|
| Speed | Fast (O(1) per query after corpus pre-encoding) | Slow (O(N) per query) |
| Accuracy | Good | Better |
| Use case | Large-scale retrieval | Re-ranking a small candidate set |

**Best practice (RAG pipeline)**:
1. **Index**: encode all documents with SBERT → store vectors
2. **Retrieve**: cosine search → top-K candidates (fast, approximate)
3. **Re-rank**: cross-encoder on top-K → final ranked list (slow, accurate)
4. **Generate**: feed re-ranked context to an LLM

**Useful SBERT models** (from [sbert.net](https://www.sbert.net/docs/sentence_transformer/pretrained_models.html)):

| Model | Dim | Speed | Quality |
|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | Very fast | Good |
| `all-MiniLM-L12-v2` | 384 | Fast | Better |
| `all-mpnet-base-v2` | 768 | Moderate | Best general |
| `multi-qa-mpnet-base-dot-v1` | 768 | Moderate | Best for QA |
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | Fast | 50+ languages |